# Étape 3 : l'audit de qualité

Avant de nettoyer, on **mesure**. Pour chacun des 9 fichiers, on cherche les problèmes de cinq familles :

| Famille | Question posée |
|---|---|
| Valeurs manquantes | quelles colonnes sont vides, et est-ce normal ? |
| Doublons | des lignes ou des clés apparaissent-elles plusieurs fois ? |
| Cohérence temporelle | les dates se suivent-elles dans un ordre logique ? |
| Cohérence entre fichiers | les identifiants et montants concordent-ils d'une table à l'autre ? |
| Valeurs aberrantes | des valeurs sont-elles impossibles ou extrêmes ? |

Chaque constat est enregistré dans un registre (`constat(...)`), résumé à la fin dans un tableau unique.
**On ne corrige rien ici** : les règles de nettoyage sont l'objet de l'étape 4. Les conclusions sont reportées dans `Carnet.md`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

DATA = next(p for p in [Path("data"), Path("../data")] if p.exists())

fichiers = {
    "customers":   "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "orders":      "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments":    "olist_order_payments_dataset.csv",
    "reviews":     "olist_order_reviews_dataset.csv",
    "products":    "olist_products_dataset.csv",
    "sellers":     "olist_sellers_dataset.csv",
    "translation": "product_category_name_translation.csv",
}
# Codes postaux en texte (identifiants, zéro initial à conserver)
zip_cols = {c: str for c in ["customer_zip_code_prefix", "seller_zip_code_prefix", "geolocation_zip_code_prefix"]}
dfs = {nom: pd.read_csv(DATA / f, dtype=zip_cols) for nom, f in fichiers.items()}
globals().update(dfs)

# Dates converties pour les contrôles temporels (les fichiers bruts restent intacts)
DATES_ORDERS = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
                "order_delivered_customer_date", "order_estimated_delivery_date"]
orders[DATES_ORDERS] = orders[DATES_ORDERS].apply(pd.to_datetime)
order_items["shipping_limit_date"] = pd.to_datetime(order_items.shipping_limit_date)
reviews[["review_creation_date", "review_answer_timestamp"]] = (
    reviews[["review_creation_date", "review_answer_timestamp"]].apply(pd.to_datetime))

pd.Series({n: d.shape for n, d in dfs.items()}, name="(lignes, colonnes)")

customers        (99441, 5)
geolocation    (1000163, 5)
orders           (99441, 8)
order_items     (112650, 7)
payments        (103886, 5)
reviews          (99224, 7)
products         (32951, 9)
sellers           (3095, 4)
translation         (71, 2)
Name: (lignes, colonnes), dtype: object

In [2]:
# Registre des constats : une ligne par problème détecté
REGISTRE = []

def constat(fichier, famille, controle, n, total=None, exemple=None):
    """Enregistre un contrôle et renvoie le nombre de lignes concernées."""
    n = int(n)
    total = len(dfs[fichier]) if total is None else total
    REGISTRE.append({"fichier": fichier, "famille": famille, "contrôle": controle,
                     "lignes": n, "%": round(100 * n / total, 2) if total else np.nan,
                     "exemple": exemple})
    return n

def manquants(fichier):
    """Valeurs manquantes par colonne + enregistrement des colonnes non complètes."""
    df = dfs[fichier]
    t = pd.DataFrame({"manquants": df.isna().sum(), "%": (df.isna().mean() * 100).round(2)})
    for col, n in t.manquants[t.manquants > 0].items():
        constat(fichier, "Valeurs manquantes", f"{col} vide", n)
    return t

def doublons(fichier, cle):
    """Doublons complets et doublons sur la clé attendue."""
    df = dfs[fichier]
    res = {"doublons complets": constat(fichier, "Doublons", "ligne entièrement dupliquée", df.duplicated().sum())}
    if cle:
        res[f"doublons sur {cle}"] = constat(fichier, "Doublons", f"clé {cle} répétée", df.duplicated(cle).sum())
    return pd.Series(res)

def iqr(s, k=1.5):
    """Bornes de Tukey : [Q1 - k·IQR, Q3 + k·IQR]."""
    q1, q3 = s.quantile([0.25, 0.75])
    return q1 - k * (q3 - q1), q3 + k * (q3 - q1)

def aberrants(fichier, col, k=3):
    """Compte les valeurs hors bornes de Tukey (k=3 : valeurs « extrêmes », pas simplement grandes)."""
    s = dfs[fichier][col].dropna()
    bas, haut = iqr(s, k)
    n = ((s < bas) | (s > haut)).sum()
    constat(fichier, "Valeurs aberrantes", f"{col} hors [Q1-{k}·IQR, Q3+{k}·IQR] = [{bas:.1f}, {haut:.1f}]", n,
            exemple=f"max = {s.max():g}")
    return pd.Series({"min": s.min(), "borne basse": bas, "borne haute": haut, "max": s.max(), "hors bornes": n})

def absents(fichier, famille, controle, valeurs, reference):
    """Valeurs d'une colonne introuvables dans une autre table (intégrité référentielle)."""
    masque = ~valeurs.isin(reference)
    return constat(fichier, famille, controle, masque.sum(),
                   exemple=valeurs[masque].iloc[0] if masque.any() else None)

In [3]:
# Contexte utile aux contrôles croisés : la date d'achat et le statut de chaque commande
info_cmd = orders.set_index("order_id")[["order_purchase_timestamp", "order_status"]]
# Préfixe postal → état(s) d'après geolocation
etats_par_cp = geolocation.groupby("geolocation_zip_code_prefix").geolocation_state.agg(lambda s: set(s))
# Ville suspecte : chiffre, « / », « , », parenthèse, « - » entouré d'espaces, ou caractère non ASCII
# (les tirets et apostrophes simples sont normaux : « varre-sai », « santa barbara d'oeste »)
VILLE_SUSPECTE = r"[0-9/,()´]| - |[^\x00-\x7f]"
ETATS_BR = {"AC","AL","AP","AM","BA","CE","DF","ES","GO","MA","MT","MS","MG","PA","PB","PR",
            "PE","PI","RJ","RN","RS","RO","RR","SC","SP","SE","TO"}

## 1. orders

Table centrale : une ligne par commande, un statut et cinq dates.

In [4]:
manquants("orders")

,manquants,%
order_id,0,0.00
customer_id,0,0.00
order_status,0,0.00
order_purchase_timestamp,0,0.00
order_approved_at,160,0.16
order_delivered_carrier_date,1783,1.79
order_delivered_customer_date,2965,2.98
order_estimated_delivery_date,0,0.00


In [5]:
# Les dates manquantes sont-elles logiques vu le statut ?
orders.groupby("order_status")[DATES_ORDERS].apply(lambda d: d.isna().sum()).assign(
    commandes=orders.order_status.value_counts())

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,commandes
order_status,,,,,,
approved,0,0,2,2,0,2
canceled,0,141,550,619,0,625
created,0,5,5,5,0,5
delivered,0,14,2,8,0,96478
invoiced,0,0,314,314,0,314
processing,0,0,301,301,0,301
shipped,0,0,0,1107,0,1107
unavailable,0,0,609,609,0,609


In [6]:
# Manquants anormaux : une commande livrée doit avoir toutes ses dates
livree = orders.order_status == "delivered"
pd.Series({
    "livrée sans date de validation": constat("orders", "Valeurs manquantes", "statut delivered sans order_approved_at",
                                             (livree & orders.order_approved_at.isna()).sum()),
    "livrée sans remise transporteur": constat("orders", "Valeurs manquantes", "statut delivered sans order_delivered_carrier_date",
                                              (livree & orders.order_delivered_carrier_date.isna()).sum()),
    "livrée sans date de livraison": constat("orders", "Valeurs manquantes", "statut delivered sans order_delivered_customer_date",
                                            (livree & orders.order_delivered_customer_date.isna()).sum()),
})

livrée sans date de validation     14
livrée sans remise transporteur     2
livrée sans date de livraison       8
dtype: int64

In [7]:
doublons("orders", ["order_id"])

doublons complets            0
doublons sur ['order_id']    0
dtype: int64

In [8]:
# Une commande = un customer_id ?
constat("orders", "Doublons", "customer_id utilisé par plusieurs commandes", orders.duplicated("customer_id").sum())

0

### Cohérence temporelle

Ordre attendu : achat ≤ validation ≤ remise au transporteur ≤ livraison, et estimation ≥ achat.

In [9]:
o = orders
regles_temps = {
    "validée avant l'achat": o.order_approved_at < o.order_purchase_timestamp,
    "remise transporteur avant l'achat": o.order_delivered_carrier_date < o.order_purchase_timestamp,
    "remise transporteur avant la validation": o.order_delivered_carrier_date < o.order_approved_at,
    "livrée avant la remise transporteur": o.order_delivered_customer_date < o.order_delivered_carrier_date,
    "livrée avant l'achat": o.order_delivered_customer_date < o.order_purchase_timestamp,
    "livraison estimée avant l'achat": o.order_estimated_delivery_date < o.order_purchase_timestamp,
    "date de livraison mais statut ≠ delivered": (o.order_status != "delivered") & o.order_delivered_customer_date.notna(),
    "remise transporteur mais statut avant expédition": o.order_status.isin(["created", "approved", "invoiced", "processing"])
                                                       & o.order_delivered_carrier_date.notna(),
}
pd.Series({k: constat("orders", "Cohérence temporelle", k, m.sum()) for k, m in regles_temps.items()}, name="lignes")

validée avant l'achat                                  0
remise transporteur avant l'achat                    166
remise transporteur avant la validation             1359
livrée avant la remise transporteur                   23
livrée avant l'achat                                   0
livraison estimée avant l'achat                        0
date de livraison mais statut ≠ delivered              6
remise transporteur mais statut avant expédition       0
Name: lignes, dtype: int64

In [10]:
# Ampleur des inversions « remise transporteur avant validation » : quelques heures ou plusieurs jours ?
m = regles_temps["remise transporteur avant la validation"]
((o.order_approved_at - o.order_delivered_carrier_date)[m].dt.total_seconds() / 3600).describe().round(1)

count    1359.0
mean       24.8
std       115.3
min         0.0
25%         1.4
50%        17.2
75%        26.0
max      4109.3
dtype: float64

In [11]:
# Couverture temporelle : mois quasi vides au début et à la fin de l'extraction
par_mois = o.order_purchase_timestamp.dt.to_period("M").value_counts().sort_index()
mois_faibles = par_mois[par_mois < 0.1 * par_mois.median()]
constat("orders", "Cohérence temporelle", f"commandes dans un mois quasi vide (< 10 % de la médiane mensuelle) : "
        f"{', '.join(map(str, mois_faibles.index))}", mois_faibles.sum())
print("période :", o.order_purchase_timestamp.min(), "→", o.order_purchase_timestamp.max())
mois_faibles

période : 2016-09-04 21:15:19 → 2018-10-17 17:30:18


order_purchase_timestamp
2016-09      4
2016-10    324
2016-12      1
2018-09     16
2018-10      4
Freq: M, Name: count, dtype: int64

In [12]:
# Commandes « bloquées » : non livrées, non annulées, et achetées longtemps avant la fin de l'extraction
fin = o.order_purchase_timestamp.max()
bloquees = ~o.order_status.isin(["delivered", "canceled", "unavailable"]) & (fin - o.order_purchase_timestamp > pd.Timedelta(days=60))
constat("orders", "Cohérence temporelle", "en cours (shipped, invoiced…) depuis plus de 60 jours à la fin de l'extraction", bloquees.sum())
o.loc[bloquees, "order_status"].value_counts()

order_status
shipped       1101
invoiced       314
processing     301
created          5
approved         2
Name: count, dtype: int64

### Cohérence entre fichiers

In [13]:
pd.Series({
    "customer_id absent de customers": absents("orders", "Cohérence entre fichiers", "customer_id absent de customers",
                                               o.customer_id, customers.customer_id),
    "commande sans article": absents("orders", "Cohérence entre fichiers", "commande sans article dans order_items",
                                     o.order_id, order_items.order_id),
    "commande sans paiement": absents("orders", "Cohérence entre fichiers", "commande sans paiement dans payments",
                                      o.order_id, payments.order_id),
    "commande sans avis": absents("orders", "Cohérence entre fichiers", "commande sans avis dans reviews",
                                  o.order_id, reviews.order_id),
})

customer_id absent de customers      0
commande sans article              775
commande sans paiement               1
commande sans avis                 768
dtype: int64

In [14]:
# Statut des commandes sans article : une commande livrée sans article est anormale
sans_article = ~o.order_id.isin(order_items.order_id)
display(o.loc[sans_article, "order_status"].value_counts())
constat("orders", "Cohérence entre fichiers", "commande delivered/shipped sans article",
        (sans_article & o.order_status.isin(["delivered", "shipped"])).sum())

order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

1

### Valeurs aberrantes

In [15]:
o_liv = o.dropna(subset=["order_delivered_customer_date"])
delais = pd.DataFrame({
    "délai livraison (j)": (o_liv.order_delivered_customer_date - o_liv.order_purchase_timestamp).dt.total_seconds() / 86400,
    "délai validation (h)": (o.order_approved_at - o.order_purchase_timestamp).dt.total_seconds() / 3600,
    "délai promis (j)": (o.order_estimated_delivery_date - o.order_purchase_timestamp).dt.total_seconds() / 86400,
})
display(delais.describe(percentiles=[.5, .9, .99]).round(1))
d = delais["délai livraison (j)"].dropna()
constat("orders", "Valeurs aberrantes", "délai de livraison > 60 jours", (d > 60).sum(), exemple=f"max = {d.max():.0f} j")
dv = delais["délai validation (h)"].dropna()
constat("orders", "Valeurs aberrantes", "validation plus de 7 jours après l'achat", (dv > 24 * 7).sum(), exemple=f"max = {dv.max()/24:.0f} j")

,délai livraison (j),délai validation (h),délai promis (j)
count,96476.0,99281.0,99441.0
mean,12.6,10.4,23.8
std,9.5,26.0,8.8
min,0.5,0.0,1.6
50%,10.2,0.3,23.2
90%,23.1,34.7,34.3
99%,46.0,90.2,50.4
max,209.6,4509.2,155.1


81

In [16]:
orders.order_status.value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

## 2. order_items

Une ligne par article : prix, frais de port, vendeur et date limite d'expédition.

In [17]:
manquants("order_items")

,manquants,%
order_id,0,0.0
order_item_id,0,0.0
product_id,0,0.0
seller_id,0,0.0
shipping_limit_date,0,0.0
price,0,0.0
freight_value,0,0.0


In [18]:
doublons("order_items", ["order_id", "order_item_id"])

doublons complets                             0
doublons sur ['order_id', 'order_item_id']    0
dtype: int64

In [19]:
# order_item_id doit être une séquence 1..n sans trou dans chaque commande
seq = order_items.groupby("order_id").order_item_id.agg(["max", "count"])
constat("order_items", "Doublons", "numérotation order_item_id avec trou (max ≠ nombre de lignes)",
        (seq["max"] != seq["count"]).sum(), total=len(seq))

0

In [20]:
oi = order_items.join(info_cmd, on="order_id")
pd.Series({
    "date limite d'expédition avant l'achat": constat("order_items", "Cohérence temporelle", "shipping_limit_date avant l'achat",
                                                      (oi.shipping_limit_date < oi.order_purchase_timestamp).sum()),
    "date limite > 60 j après l'achat": constat("order_items", "Cohérence temporelle", "shipping_limit_date plus de 60 j après l'achat",
                                                (oi.shipping_limit_date - oi.order_purchase_timestamp > pd.Timedelta(days=60)).sum(),
                                                exemple=str((oi.shipping_limit_date - oi.order_purchase_timestamp).max())),
})

date limite d'expédition avant l'achat     0
date limite > 60 j après l'achat          12
dtype: int64

In [21]:
pd.Series({
    "order_id absent de orders": absents("order_items", "Cohérence entre fichiers", "order_id absent de orders",
                                         order_items.order_id, orders.order_id),
    "product_id absent de products": absents("order_items", "Cohérence entre fichiers", "product_id absent de products",
                                             order_items.product_id, products.product_id),
    "seller_id absent de sellers": absents("order_items", "Cohérence entre fichiers", "seller_id absent de sellers",
                                           order_items.seller_id, sellers.seller_id),
    "article d'une commande canceled/unavailable": constat("order_items", "Cohérence entre fichiers",
                                                           "article rattaché à une commande canceled/unavailable",
                                                           oi.order_status.isin(["canceled", "unavailable"]).sum()),
})

order_id absent de orders                        0
product_id absent de products                    0
seller_id absent de sellers                      0
article d'une commande canceled/unavailable    549
dtype: int64

In [22]:
display(order_items[["price", "freight_value"]].describe(percentiles=[.01, .5, .99]).round(2))
display(pd.DataFrame({c: aberrants("order_items", c) for c in ["price", "freight_value"]}).round(2))
pd.Series({
    "prix ≤ 0": constat("order_items", "Valeurs aberrantes", "price ≤ 0", (order_items.price <= 0).sum()),
    "port = 0": constat("order_items", "Valeurs aberrantes", "freight_value = 0 (livraison gratuite ?)", (order_items.freight_value == 0).sum()),
    "port > prix": constat("order_items", "Valeurs aberrantes", "freight_value > price", (order_items.freight_value > order_items.price).sum()),
})

,price,freight_value
count,112650.00,112650.00
mean,120.65,19.99
std,183.63,15.81
min,0.85,0.00
1%,9.99,4.42
50%,74.99,16.26
99%,890.00,84.52
max,6735.00,409.68


,price,freight_value
min,0.85,0.00
borne basse,-245.10,-11.13
borne haute,419.90,45.36
max,6735.00,409.68
hors bornes,4074.00,5538.00


prix ≤ 0          0
port = 0        383
port > prix    4124
dtype: int64

In [23]:
# Un même produit vendu à des prix très différents ? (rapport max/min > 10)
ecart_prix = order_items.groupby("product_id").price.agg(lambda s: s.max() / s.min())
constat("order_items", "Valeurs aberrantes", "produit vendu à des prix variant d'un facteur > 10",
        (ecart_prix > 10).sum(), total=len(ecart_prix), exemple=f"facteur max = {ecart_prix.max():.0f}")

0

## 3. payments

Un ou plusieurs paiements par commande : type, nombre d'échéances, montant.

In [24]:
manquants("payments")

,manquants,%
order_id,0,0.0
payment_sequential,0,0.0
payment_type,0,0.0
payment_installments,0,0.0
payment_value,0,0.0


In [25]:
doublons("payments", ["order_id", "payment_sequential"])

doublons complets                                  0
doublons sur ['order_id', 'payment_sequential']    0
dtype: int64

In [26]:
# payment_sequential doit aller de 1 à n
seq = payments.groupby("order_id").payment_sequential.agg(["max", "count"])
constat("payments", "Doublons", "numérotation payment_sequential avec trou (max ≠ nombre de lignes)",
        (seq["max"] != seq["count"]).sum(), total=len(seq))

80

*Cohérence temporelle : la table n'a pas de date, rien à contrôler.*

In [27]:
pay_cmd = payments.groupby("order_id").payment_value.sum()
art_cmd = order_items.groupby("order_id")[["price", "freight_value"]].sum().sum(axis=1)
ecart = (pay_cmd - art_cmd).dropna()
display(ecart.describe(percentiles=[.01, .5, .99]).round(2))
pd.Series({
    "order_id absent de orders": absents("payments", "Cohérence entre fichiers", "order_id absent de orders",
                                         payments.order_id, orders.order_id),
    "payé ≠ articles + port (écart > 1 R$)": constat("payments", "Cohérence entre fichiers",
                                                     "commande où le total payé ≠ prix + port (écart > 1 R$)",
                                                     (ecart.abs() > 1).sum(), total=len(ecart),
                                                     exemple=f"écart de {ecart.min():.0f} à {ecart.max():.0f} R$"),
    "paiement d'une commande sans article": constat("payments", "Cohérence entre fichiers",
                                                    "paiement d'une commande sans article",
                                                    (~payments.order_id.isin(order_items.order_id)).sum()),
})

count    98665.00
mean         0.03
std          1.13
min        -51.62
1%          -0.00
50%          0.00
99%          0.00
max        182.81
dtype: float64

order_id absent de orders                  0
payé ≠ articles + port (écart > 1 R$)    249
paiement d'une commande sans article     830
dtype: int64

In [28]:
display(payments.payment_type.value_counts())
display(payments.payment_installments.value_counts().sort_index().to_frame().T)
display(aberrants("payments", "payment_value").round(2).to_frame().T)
pd.Series({
    "type not_defined": constat("payments", "Valeurs aberrantes", "payment_type = not_defined", (payments.payment_type == "not_defined").sum()),
    "échéances = 0": constat("payments", "Valeurs aberrantes", "payment_installments = 0", (payments.payment_installments == 0).sum()),
    "montant = 0": constat("payments", "Valeurs aberrantes", "payment_value = 0", (payments.payment_value == 0).sum()),
    "échéances > 1 hors carte de crédit": constat("payments", "Valeurs aberrantes", "plusieurs échéances sans carte de crédit",
                                                  ((payments.payment_installments > 1) & (payments.payment_type != "credit_card")).sum()),
})

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

payment_installments,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,20,21,22,23,24
count,2,52546,12413,10461,7098,5239,3920,1626,4268,644,5328,23,133,16,15,74,5,8,27,17,3,1,1,18


,min,borne basse,borne haute,max,hors bornes
0,0.0,-288.35,516.98,13664.08,4046.0


type not_defined                      3
échéances = 0                         2
montant = 0                           9
échéances > 1 hors carte de crédit    0
dtype: int64

## 4. reviews

L'avis client : note 1–5, titre et message facultatifs, date d'envoi du questionnaire et de réponse.

In [29]:
manquants("reviews")

,manquants,%
review_id,0,0.00
order_id,0,0.00
review_score,0,0.00
review_comment_title,87656,88.34
review_comment_message,58247,58.70
review_creation_date,0,0.00
review_answer_timestamp,0,0.00


In [30]:
s = doublons("reviews", ["review_id", "order_id"])
s["review_id répété"] = constat("reviews", "Doublons", "review_id partagé par plusieurs commandes", reviews.review_id.duplicated().sum())
s["commande avec plusieurs avis"] = constat("reviews", "Doublons", "order_id avec plusieurs avis", reviews.order_id.duplicated().sum())
s

doublons complets                           0
doublons sur ['review_id', 'order_id']      0
review_id répété                          814
commande avec plusieurs avis              551
dtype: int64

In [31]:
# Une commande avec plusieurs avis : notes identiques ou contradictoires ?
plusieurs = reviews[reviews.order_id.duplicated(keep=False)]
notes_diff = plusieurs.groupby("order_id").review_score.nunique()
constat("reviews", "Doublons", "commande avec plusieurs avis de notes différentes", (notes_diff > 1).sum(), total=reviews.order_id.nunique())

202

In [32]:
rv = reviews.join(info_cmd, on="order_id")
liv = orders.set_index("order_id").order_delivered_customer_date
rv["livraison"] = rv.order_id.map(liv)
pd.Series({
    "avis créé avant l'achat": constat("reviews", "Cohérence temporelle", "review_creation_date avant le jour de l'achat",
                                       (rv.review_creation_date < rv.order_purchase_timestamp.dt.normalize()).sum()),
    "réponse avant création": constat("reviews", "Cohérence temporelle", "review_answer_timestamp avant review_creation_date",
                                      (rv.review_answer_timestamp < rv.review_creation_date).sum()),
    "avis avant la livraison": constat("reviews", "Cohérence temporelle", "avis créé avant la livraison effective",
                                       (rv.review_creation_date < rv.livraison.dt.normalize()).sum()),
})

avis créé avant l'achat      64
réponse avant création        0
avis avant la livraison    5127
dtype: int64

In [33]:
pd.Series({
    "order_id absent de orders": absents("reviews", "Cohérence entre fichiers", "order_id absent de orders",
                                         reviews.order_id, orders.order_id),
    "avis sur commande canceled/unavailable": constat("reviews", "Cohérence entre fichiers", "avis sur une commande canceled/unavailable",
                                                      rv.order_status.isin(["canceled", "unavailable"]).sum()),
})

order_id absent de orders                    0
avis sur commande canceled/unavailable    1206
dtype: int64

In [34]:
display(reviews.review_score.value_counts().sort_index().to_frame().T)
delai_rep = (reviews.review_answer_timestamp - reviews.review_creation_date).dt.total_seconds() / 86400
display(delai_rep.describe(percentiles=[.5, .99]).round(1).to_frame("délai de réponse (j)").T)
pd.Series({
    "note hors 1–5": constat("reviews", "Valeurs aberrantes", "review_score hors 1–5", (~reviews.review_score.between(1, 5)).sum()),
    "réponse > 30 j après le questionnaire": constat("reviews", "Valeurs aberrantes", "réponse plus de 30 j après le questionnaire",
                                                     (delai_rep > 30).sum(), exemple=f"max = {delai_rep.max():.0f} j"),
})

review_score,1,2,3,4,5
count,11424,3151,8179,19142,57328


,count,mean,std,min,50%,99%,max
délai de réponse (j),99224.0,3.1,9.9,0.1,1.7,22.1,518.7


note hors 1–5                              0
réponse > 30 j après le questionnaire    682
dtype: int64

## 5. products

Le catalogue : catégorie, longueurs de la fiche, nombre de photos, poids et dimensions.

In [35]:
manquants("products")

,manquants,%
product_id,0,0.00
product_category_name,610,1.85
product_name_lenght,610,1.85
product_description_lenght,610,1.85
product_photos_qty,610,1.85
product_weight_g,2,0.01
product_length_cm,2,0.01
product_height_cm,2,0.01
product_width_cm,2,0.01


In [36]:
# Les manquants se recoupent-ils ? (mêmes produits sans catégorie / sans fiche)
cols_fiche = ["product_category_name", "product_name_lenght", "product_description_lenght", "product_photos_qty"]
cols_dim = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]
products[cols_fiche + cols_dim].isna().value_counts().rename("produits").to_frame()

,,,,,,,,produits
product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,
False,False,False,False,False,False,False,False,32340
True,True,True,True,False,False,False,False,609
False,False,False,False,True,True,True,True,1
True,True,True,True,True,True,True,True,1


In [37]:
doublons("products", ["product_id"])

doublons complets              0
doublons sur ['product_id']    0
dtype: int64

*Cohérence temporelle : pas de date dans la table.*

In [38]:
pd.Series({
    "catégorie sans traduction": absents("products", "Cohérence entre fichiers", "produit dont la catégorie est absente de translation",
                                         products.product_category_name.dropna(), translation.product_category_name),
    "produit jamais vendu": absents("products", "Cohérence entre fichiers", "produit absent de order_items",
                                    products.product_id, order_items.product_id),
})

catégorie sans traduction    13
produit jamais vendu          0
dtype: int64

In [39]:
display(products.describe().round(1))
display(pd.DataFrame({c: aberrants("products", c) for c in cols_dim}).round(1))
volume_l = products.product_length_cm * products.product_height_cm * products.product_width_cm / 1000
densite = products.product_weight_g / (volume_l * 1000)  # g/cm³
pd.Series({
    "poids = 0 g": constat("products", "Valeurs aberrantes", "product_weight_g = 0", (products.product_weight_g == 0).sum()),
    "densité > 10 g/cm³ (plus lourd que le fer)": constat("products", "Valeurs aberrantes", "densité > 10 g/cm³ (poids incohérent avec le volume)",
                                                            (densite > 10).sum(), exemple=f"max = {densite.max():.1f} g/cm³"),
    "0 photo": constat("products", "Valeurs aberrantes", "product_photos_qty = 0", (products.product_photos_qty == 0).sum()),
})

,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32341.0,32341.0,32341.0,32949.0,32949.0,32949.0,32949.0
mean,48.5,771.5,2.2,2276.5,30.8,16.9,23.2
std,10.2,635.1,1.7,4282.0,16.9,13.6,12.1
min,5.0,4.0,1.0,0.0,7.0,2.0,6.0
25%,42.0,339.0,1.0,300.0,18.0,8.0,15.0
50%,51.0,595.0,1.0,700.0,25.0,13.0,20.0
75%,57.0,972.0,3.0,1900.0,38.0,21.0,30.0
max,76.0,3992.0,20.0,40425.0,105.0,105.0,118.0


,product_weight_g,product_length_cm,product_height_cm,product_width_cm
min,0.0,7.0,2.0,6.0
borne basse,-4500.0,-42.0,-31.0,-30.0
borne haute,6700.0,98.0,60.0,75.0
max,40425.0,105.0,105.0,118.0
hors bornes,3073.0,349.0,559.0,108.0


poids = 0 g                                    4
densité > 10 g/cm³ (plus lourd que le fer)    18
0 photo                                        0
dtype: int64

## 6. customers

Une ligne par `customer_id` (donc par commande), avec l'identifiant de la personne et sa localisation.

In [40]:
manquants("customers")

,manquants,%
customer_id,0,0.0
customer_unique_id,0,0.0
customer_zip_code_prefix,0,0.0
customer_city,0,0.0
customer_state,0,0.0


In [41]:
s = doublons("customers", ["customer_id"])
# customer_unique_id répété : attendu (clients fidèles), mais la personne change-t-elle d'adresse ?
loc = customers.groupby("customer_unique_id")[["customer_zip_code_prefix", "customer_state"]].nunique()
s["personnes avec plusieurs codes postaux"] = constat("customers", "Doublons", "même personne avec plusieurs codes postaux",
                                                      (loc.customer_zip_code_prefix > 1).sum(), total=len(loc))
s["personnes avec plusieurs États"] = constat("customers", "Doublons", "même personne dans plusieurs États",
                                              (loc.customer_state > 1).sum(), total=len(loc))
s

doublons complets                           0
doublons sur ['customer_id']                0
personnes avec plusieurs codes postaux    250
personnes avec plusieurs États             39
dtype: int64

In [42]:
# Cohérence temporelle (via orders) : une même personne passe-t-elle deux commandes à la même seconde ?
cmd_pers = orders.merge(customers[["customer_id", "customer_unique_id"]], on="customer_id")
constat("customers", "Cohérence temporelle", "personne avec deux commandes au même instant",
        cmd_pers.duplicated(["customer_unique_id", "order_purchase_timestamp"]).sum())

292

In [43]:
etat_attendu = customers.customer_zip_code_prefix.map(etats_par_cp)
incoherent = [isinstance(e, set) and s not in e for s, e in zip(customers.customer_state, etat_attendu)]
pd.Series({
    "customer_id sans commande": absents("customers", "Cohérence entre fichiers", "customer_id absent de orders",
                                         customers.customer_id, orders.customer_id),
    "code postal absent de geolocation": absents("customers", "Cohérence entre fichiers", "code postal absent de geolocation",
                                                 customers.customer_zip_code_prefix, geolocation.geolocation_zip_code_prefix),
    "État ≠ État du code postal": constat("customers", "Cohérence entre fichiers", "customer_state différent de l'État du code postal (geolocation)",
                                          sum(incoherent)),
})

customer_id sans commande              0
code postal absent de geolocation    278
État ≠ État du code postal             0
dtype: int64

In [44]:
cp = customers.customer_zip_code_prefix
pd.Series({
    "code postal ≠ 5 chiffres": constat("customers", "Valeurs aberrantes", "code postal qui n'a pas 5 chiffres", (~cp.str.fullmatch(r"\d{5}")).sum()),
    "État inconnu": constat("customers", "Valeurs aberrantes", "État hors des 27 UF", (~customers.customer_state.isin(ETATS_BR)).sum()),
    "ville mal saisie": constat("customers", "Valeurs aberrantes", "ville contenant chiffre, accent, « / », « , »…",
                                customers.customer_city.str.contains(VILLE_SUSPECTE).sum()),
})

code postal ≠ 5 chiffres    0
État inconnu                0
ville mal saisie            1
dtype: int64

## 7. sellers

Les vendeurs et leur localisation.

In [45]:
manquants("sellers")

,manquants,%
seller_id,0,0.0
seller_zip_code_prefix,0,0.0
seller_city,0,0.0
seller_state,0,0.0


In [46]:
doublons("sellers", ["seller_id"])

doublons complets             0
doublons sur ['seller_id']    0
dtype: int64

In [47]:
# Cohérence temporelle (via order_items) : période d'activité de chaque vendeur
# Pas de date dans sellers : on regarde seulement la période d'activité (information, pas un défaut)
act = oi.groupby("seller_id").order_purchase_timestamp.agg(["min", "max", "count"])
print("vendeurs actifs un seul jour :", (act["min"].dt.date == act["max"].dt.date).sum())
act.describe().T

vendeurs actifs un seul jour : 579


,count,mean,min,25%,50%,75%,max,std
min,3095,2017-10-20 23:43:59.423909,2016-09-04 21:15:19,2017-04-22 14:58:59.500000,2017-11-04 19:22:13,2018-04-19 19:46:24.500000,2018-08-28 09:26:43,NaN
max,3095,2018-04-15 05:49:27.091438,2016-10-03 16:56:50,2018-01-25 01:15:42.500000,2018-07-14 14:08:11,2018-08-16 11:35:52,2018-09-03 09:06:57,NaN
count,3095.0,36.397415,1.0,2.0,8.0,24.0,2033.0,119.193461


In [48]:
etat_attendu = sellers.seller_zip_code_prefix.map(etats_par_cp)
incoherent = [isinstance(e, set) and s not in e for s, e in zip(sellers.seller_state, etat_attendu)]
pd.Series({
    "vendeur sans vente": absents("sellers", "Cohérence entre fichiers", "seller_id absent de order_items",
                                  sellers.seller_id, order_items.seller_id),
    "code postal absent de geolocation": absents("sellers", "Cohérence entre fichiers", "code postal absent de geolocation",
                                                 sellers.seller_zip_code_prefix, geolocation.geolocation_zip_code_prefix),
    "État ≠ État du code postal": constat("sellers", "Cohérence entre fichiers", "seller_state différent de l'État du code postal (geolocation)",
                                          sum(incoherent)),
})

vendeur sans vente                    0
code postal absent de geolocation     7
État ≠ État du code postal           35
dtype: int64

In [49]:
villes = sellers.seller_city
bizarres = villes[villes.str.contains(VILLE_SUSPECTE)]
display(bizarres.value_counts().head(15))
pd.Series({
    "code postal ≠ 5 chiffres": constat("sellers", "Valeurs aberrantes", "code postal qui n'a pas 5 chiffres",
                                        (~sellers.seller_zip_code_prefix.str.fullmatch(r"\d{5}")).sum()),
    "État inconnu": constat("sellers", "Valeurs aberrantes", "État hors des 27 UF", (~sellers.seller_state.isin(ETATS_BR)).sum()),
    "ville mal saisie": constat("sellers", "Valeurs aberrantes", "ville contenant chiffre, accent, « / », « , »…",
                                len(bizarres), exemple=" | ".join(bizarres.unique()[:4])),
})

seller_city
sao paulo - sp                              3
santa barbara d´oeste                       2
lages - sc                                  1
auriflama/sp                                1
sao paulo / sao paulo                       1
são paulo                                  1
04482255                                    1
novo hamburgo, rio grande do sul, brasil    1
cariacica / es                              1
sbc/sp                                      1
arraial d'ajuda (porto seguro)              1
santo andre/sao paulo                       1
sp / sp                                     1
maua/sao paulo                              1
mogi das cruzes / sp                        1
Name: count, dtype: int64

code postal ≠ 5 chiffres     0
État inconnu                 0
ville mal saisie            26
dtype: int64

In [50]:
# Variantes d'écriture d'une même ville une fois normalisée (minuscules, sans accent, sans « / état »)
import unicodedata
def normalise(v):
    v = unicodedata.normalize("NFKD", str(v)).encode("ascii", "ignore").decode().lower()
    return v.split("/")[0].split(",")[0].split(" - ")[0].strip()
variantes = sellers.assign(norm=villes.map(normalise)).groupby("norm").seller_city.nunique()
constat("sellers", "Valeurs aberrantes", "ville écrite de plusieurs façons", (variantes > 1).sum(), total=variantes.size,
        exemple=variantes.idxmax())
variantes[variantes > 1].sort_values(ascending=False).head()

norm
sao paulo         4
rio de janeiro    3
auriflama         2
barbacena         2
carapicuiba       2
Name: seller_city, dtype: int64

## 8. geolocation

Des points GPS par préfixe postal (plusieurs par préfixe, pas de clé).

In [51]:
manquants("geolocation")

,manquants,%
geolocation_zip_code_prefix,0,0.0
geolocation_lat,0,0.0
geolocation_lng,0,0.0
geolocation_city,0,0.0
geolocation_state,0,0.0


In [52]:
s = doublons("geolocation", None)
pts = geolocation.groupby("geolocation_zip_code_prefix").size()
s["préfixes avec plusieurs points"] = constat("geolocation", "Doublons", "préfixe postal avec plusieurs lignes (pas de clé)",
                                              (pts > 1).sum(), total=pts.size, exemple=f"jusqu'à {pts.max()} points")
s

doublons complets                 261831
préfixes avec plusieurs points     17972
dtype: int64

*Cohérence temporelle : pas de date dans la table.*

In [53]:
g = geolocation
etats = g.groupby("geolocation_zip_code_prefix").geolocation_state.nunique()
pd.Series({
    "préfixe dans plusieurs États": constat("geolocation", "Cohérence entre fichiers", "préfixe postal rattaché à plusieurs États",
                                            (etats > 1).sum(), total=etats.size),
    "préfixe jamais utilisé": constat("geolocation", "Cohérence entre fichiers", "préfixe utilisé ni par un client ni par un vendeur",
                                      (~etats.index.isin(set(customers.customer_zip_code_prefix) | set(sellers.seller_zip_code_prefix))).sum(),
                                      total=etats.size),
})

préfixe dans plusieurs États       8
préfixe jamais utilisé          4099
dtype: int64

In [54]:
hors_bresil = ~g.geolocation_lat.between(-34, 6) | ~g.geolocation_lng.between(-74, -34)
# Point éloigné de la médiane de son préfixe (> 50 km ≈ 0,45°) : préfixe mal géocodé
med = g.groupby("geolocation_zip_code_prefix")[["geolocation_lat", "geolocation_lng"]].transform("median")
dist_km = np.hypot(g.geolocation_lat - med.geolocation_lat,
                   (g.geolocation_lng - med.geolocation_lng) * np.cos(np.radians(med.geolocation_lat))) * 111
villes_ecrites = g.groupby("geolocation_zip_code_prefix").geolocation_city.nunique()
pd.Series({
    "point hors du Brésil": constat("geolocation", "Valeurs aberrantes", "point hors de la boîte du Brésil (lat −34..6, lng −74..−34)",
                                    hors_bresil.sum()),
    "point à > 50 km de son préfixe": constat("geolocation", "Valeurs aberrantes", "point à plus de 50 km de la médiane de son préfixe",
                                              (dist_km > 50).sum(), exemple=f"max = {dist_km.max():.0f} km"),
    "ville écrite de plusieurs façons": constat("geolocation", "Valeurs aberrantes", "préfixe avec plusieurs orthographes de ville",
                                                (villes_ecrites > 1).sum(), total=villes_ecrites.size,
                                                exemple=" / ".join(g[g.geolocation_zip_code_prefix == villes_ecrites.idxmax()].geolocation_city.unique()[:3])),
})

point hors du Brésil                  42
point à > 50 km de son préfixe      1186
ville écrite de plusieurs façons    8556
dtype: int64

## 9. product_category_name_translation

Dictionnaire portugais → anglais des catégories.

In [55]:
manquants("translation")

,manquants,%
product_category_name,0,0.0
product_category_name_english,0,0.0


In [56]:
s = doublons("translation", ["product_category_name"])
s["traduction anglaise répétée"] = constat("translation", "Doublons", "même traduction anglaise pour plusieurs catégories",
                                           translation.product_category_name_english.duplicated().sum())
s

doublons complets                         0
doublons sur ['product_category_name']    0
traduction anglaise répétée               0
dtype: int64

*Cohérence temporelle : pas de date dans la table.*

In [57]:
pd.Series({
    "catégorie traduite jamais utilisée": absents("translation", "Cohérence entre fichiers", "catégorie absente de products",
                                                  translation.product_category_name, products.product_category_name),
    "catégorie de products non traduite": len(set(products.product_category_name.dropna()) - set(translation.product_category_name)),
})

catégorie traduite jamais utilisée    0
catégorie de products non traduite    2
dtype: int64

In [58]:
# Valeurs aberrantes : noms mal formés (majuscules, espaces, caractères inattendus)
noms = pd.concat([translation.product_category_name, translation.product_category_name_english])
constat("translation", "Valeurs aberrantes", "nom de catégorie hors [a-z0-9_]", (~noms.str.fullmatch(r"[a-z0-9_]+")).sum(),
        total=len(translation))

0

## 10. client_geo (PostgreSQL)

Table fournie : une position par `customer_id`. On vérifie qu'elle couvre bien tous les clients.

In [59]:
try:
    from sqlalchemy import create_engine
    client_geo = pd.read_sql("SELECT * FROM client_geo", create_engine("postgresql+psycopg2://"))
    dfs["client_geo"] = client_geo
    manquants("client_geo")
    doublons("client_geo", ["customer_id"])
    absents("client_geo", "Cohérence entre fichiers", "customer_id absent de customers", client_geo.customer_id, customers.customer_id)
    absents("customers", "Cohérence entre fichiers", "customer_id absent de client_geo", customers.customer_id, client_geo.customer_id)
    display(client_geo[["latitude", "longitude"]].astype(float).describe().round(2))
except Exception as e:
    print("client_geo non disponible :", e)

,latitude,longitude
count,99441.00,99441.00
mean,-21.17,-46.17
std,5.60,4.06
min,-33.69,-72.68
25%,-23.55,-47.99
50%,-22.92,-46.63
75%,-20.09,-43.48
max,3.84,-34.80


## 11. Synthèse de l'audit

Tous les contrôles, y compris ceux qui ne trouvent rien (0 ligne) : un contrôle à 0 est aussi une information.

In [60]:
audit = pd.DataFrame(REGISTRE)
problemes = audit[audit.lignes > 0]
print(f"{len(audit)} contrôles, dont {len(problemes)} avec au moins une ligne concernée")
pd.crosstab(problemes.fichier, problemes.famille, margins=True, margins_name="total")

121 contrôles, dont 71 avec au moins une ligne concernée


famille,Cohérence entre fichiers,Cohérence temporelle,Doublons,Valeurs aberrantes,Valeurs manquantes,total
fichier,,,,,,
customers,1,1,2,1,0,5
geolocation,2,0,2,3,0,7
order_items,1,1,0,4,0,6
orders,4,6,0,2,6,18
payments,2,0,1,4,0,7
products,1,0,0,6,8,15
reviews,1,2,3,1,2,9
sellers,2,0,0,2,0,4
total,14,10,8,23,16,71


In [61]:
with pd.option_context("display.max_rows", 200):
    display(problemes.sort_values(["fichier", "famille", "lignes"], ascending=[True, True, False]).reset_index(drop=True))

,fichier,famille,contrôle,lignes,%,exemple
0,customers,Cohérence entre fichiers,code postal absent de geolocation,278,0.28,72300
1,customers,Cohérence temporelle,personne avec deux commandes au même instant,292,0.29,NaN
2,customers,Doublons,même personne avec plusieurs codes postaux,250,0.26,NaN
3,customers,Doublons,même personne dans plusieurs États,39,0.04,NaN
4,customers,Valeurs aberrantes,"ville contenant chiffre, accent, « / », « , »…",1,0.00,NaN
5,geolocation,Cohérence entre fichiers,préfixe utilisé ni par un client ni par un vendeur,4099,21.56,NaN
6,geolocation,Cohérence entre fichiers,préfixe postal rattaché à plusieurs États,8,0.04,NaN
7,geolocation,Doublons,ligne entièrement dupliquée,261831,26.18,NaN
8,geolocation,Doublons,préfixe postal avec plusieurs lignes (pas de clé),17972,94.51,jusqu'à 1146 points
9,geolocation,Valeurs aberrantes,préfixe avec plusieurs orthographes de ville,8556,45.00,embu-guacu / embu-guaçu / embu guaçu


In [62]:
# Contrôles sans aucun problème
audit.loc[audit.lignes == 0, ["fichier", "famille", "contrôle"]].reset_index(drop=True)

,fichier,famille,contrôle
0,orders,Doublons,ligne entièrement dupliquée
1,orders,Doublons,clé ['order_id'] répétée
2,orders,Doublons,customer_id utilisé par plusieurs commandes
3,orders,Cohérence temporelle,validée avant l'achat
4,orders,Cohérence temporelle,livrée avant l'achat
5,orders,Cohérence temporelle,livraison estimée avant l'achat
6,orders,Cohérence temporelle,remise transporteur mais statut avant expédition
7,orders,Cohérence entre fichiers,customer_id absent de customers
8,order_items,Doublons,ligne entièrement dupliquée
9,order_items,Doublons,"clé ['order_id', 'order_item_id'] répétée"
